# Section 2: The Exploitability Overlay — Rain vs. Floods

**ITESO | Vulnerability Management and the AI Crossroads**

This notebook builds a real exploitability overlay by combining three live data sources:
- **CISA KEV catalog** — confirmed in-the-wild exploits
- **EPSS API** — probability of exploitation in the next 30 days
- **NVD API** — CVSS severity scores for context

**What we'll demonstrate:**
1. Download and analyse the full CISA KEV catalog
2. Fetch EPSS scores and compare their distribution to CVSS
3. Build a real triage query: given a CVE list, rank by exploitability
4. Compute the noise-to-signal ratio at each patching tier

> **Key insight:** ~40,000 CVEs were published in 2024. CISA added ~200 to KEV. That's a 200:1 noise-to-signal ratio — before you even open a ticket.

In [ ]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)

KEV_URL  = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
EPSS_URL = 'https://api.first.org/data/v1/epss'
NVD_API  = 'https://services.nvd.nist.gov/rest/json/cves/2.0'

NVD_API_KEY = None  # optional

print('Setup complete.')

## 1. Downloading and Analysing the CISA KEV Catalog

The KEV catalog is a single JSON file maintained by CISA. Every entry is a CVE that has been confirmed as actively exploited in the wild.

In [ ]:
KEV_CACHE = OUTPUT / 'kev_catalog.json'

if KEV_CACHE.exists():
    with open(KEV_CACHE) as f:
        kev_raw = json.load(f)
    print(f'Loaded KEV from cache ({KEV_CACHE})')
else:
    print('Downloading KEV catalog...')
    r = requests.get(KEV_URL, timeout=30)
    r.raise_for_status()
    kev_raw = r.json()
    with open(KEV_CACHE, 'w') as f:
        json.dump(kev_raw, f)
    print('Downloaded and cached.')

kev = pd.DataFrame(kev_raw['vulnerabilities'])
kev['dateAdded'] = pd.to_datetime(kev['dateAdded'])
kev['year_added'] = kev['dateAdded'].dt.year

print(f'\nKEV catalog: {len(kev):,} total entries')
print(f'Date range:  {kev["dateAdded"].min().date()} → {kev["dateAdded"].max().date()}')
print(f'\nEntries added per year:')
print(kev.groupby('year_added').size().to_string())

In [ ]:
# Top vendors by KEV count
print('Top 15 vendors by KEV entries:')
vendor_counts = kev['vendorProject'].value_counts().head(15)
for vendor, count in vendor_counts.items():
    bar = '█' * (count // 3)
    print(f'  {vendor:<25} {count:>4}  {bar}')

## 2. EPSS Scores for KEV Entries

Now let's fetch EPSS scores for KEV CVEs. This answers: **do the CVEs that are actively exploited actually have high EPSS scores?** (If yes, EPSS is a good predictive filter.)

In [ ]:
EPSS_CACHE = OUTPUT / 'epss_kev_scores.json'

def fetch_epss_batch(cve_ids: list) -> dict:
    """Fetch EPSS scores for a list of CVE IDs. Returns {cve_id: score}."""
    results = {}
    # EPSS API accepts comma-separated CVE IDs (up to ~100 per request)
    batch_size = 100
    for i in range(0, len(cve_ids), batch_size):
        batch = cve_ids[i:i + batch_size]
        params = {'cve': ','.join(batch)}
        try:
            r = requests.get(EPSS_URL, params=params, timeout=20)
            r.raise_for_status()
            for entry in r.json().get('data', []):
                results[entry['cve']] = float(entry['epss'])
        except Exception as e:
            print(f'  Batch {i//batch_size + 1} failed: {e}')
        time.sleep(0.5)
    return results


if EPSS_CACHE.exists():
    with open(EPSS_CACHE) as f:
        epss_kev = json.load(f)
    print(f'Loaded EPSS cache: {len(epss_kev):,} scores')
else:
    print(f'Fetching EPSS scores for {len(kev):,} KEV CVEs...')
    epss_kev = fetch_epss_batch(kev['cveID'].tolist())
    with open(EPSS_CACHE, 'w') as f:
        json.dump(epss_kev, f)
    print(f'Fetched {len(epss_kev):,} scores.')

kev['epss'] = kev['cveID'].map(epss_kev)
kev_with_epss = kev.dropna(subset=['epss'])

print(f'\nKEV entries with EPSS scores: {len(kev_with_epss):,}')
print(f'\nEPSS score distribution for KEV entries:')
print(kev_with_epss['epss'].describe().to_string())
print(f'\n% of KEV entries with EPSS > 10%: {(kev_with_epss["epss"] > 0.10).mean()*100:.1f}%')
print(f'% of KEV entries with EPSS > 50%: {(kev_with_epss["epss"] > 0.50).mean()*100:.1f}%')

## 3. CVSS vs. EPSS — Two Different Questions

CVSS asks: *"How bad could this be?"*  
EPSS asks: *"How likely is this to actually appear in an attack?"*

They are not the same — and the scatter plot below shows that clearly.

In [ ]:
# Fetch CVSS scores for KEV entries from NVD (sample for speed)
CVSS_CACHE = OUTPUT / 'cvss_kev_sample.json'

if CVSS_CACHE.exists():
    with open(CVSS_CACHE) as f:
        cvss_data = json.load(f)
    print(f'Loaded CVSS cache: {len(cvss_data):,} entries')
else:
    print('Fetching CVSS scores from NVD (sampling 200 KEV entries)...')
    sample_cves = kev_with_epss.sample(min(200, len(kev_with_epss)), random_state=42)['cveID'].tolist()
    cvss_data = {}
    headers = {'apiKey': NVD_API_KEY} if NVD_API_KEY else {}
    delay = 0.6 if NVD_API_KEY else 7
    for cve_id in sample_cves:
        try:
            r = requests.get(NVD_API, params={'cveId': cve_id}, headers=headers, timeout=15)
            r.raise_for_status()
            vulns = r.json().get('vulnerabilities', [])
            if vulns:
                metrics = vulns[0]['cve'].get('metrics', {})
                for version in ['cvssMetricV31', 'cvssMetricV30', 'cvssMetricV2']:
                    if version in metrics:
                        cvss_data[cve_id] = metrics[version][0]['cvssData']['baseScore']
                        break
        except Exception:
            pass
        time.sleep(delay)
    with open(CVSS_CACHE, 'w') as f:
        json.dump(cvss_data, f)
    print(f'Fetched {len(cvss_data):,} CVSS scores.')

kev['cvss'] = kev['cveID'].map(cvss_data)
scatter_df = kev.dropna(subset=['epss', 'cvss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.subplots_adjust(top=0.88, bottom=0.12)

# Scatter: CVSS vs EPSS
ax = axes[0]
ax.scatter(scatter_df['cvss'], scatter_df['epss'] * 100,
           alpha=0.4, s=18, color='#4a90d9')
ax.axhline(10, color='#e06c5a', lw=1.5, linestyle='--', label='EPSS = 10%')
ax.set_xlabel('CVSS Base Score')
ax.set_ylabel('EPSS Score (%)')
ax.set_title('CVSS vs. EPSS for KEV Entries\n(High CVSS ≠ High EPSS)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

r_val = scatter_df[['cvss', 'epss']].corr().iloc[0, 1]
ax.text(0.05, 0.95, f'Pearson r = {r_val:.2f}', transform=ax.transAxes,
        fontsize=9, va='top', color='#333',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# EPSS distribution for KEV
ax2 = axes[1]
epss_vals = kev_with_epss['epss'] * 100
ax2.hist(epss_vals, bins=40, color='#4a90d9', alpha=0.8, edgecolor='white', lw=0.4)
ax2.axvline(10, color='#e06c5a', lw=2, linestyle='--', label='10% threshold')
ax2.axvline(epss_vals.median(), color='#2c2c2c', lw=1.5, linestyle='-.',
            label=f'Median = {epss_vals.median():.1f}%')
ax2.set_xlabel('EPSS Score (%)')
ax2.set_ylabel('Number of KEV Entries')
ax2.set_title('EPSS Score Distribution\nFor All KEV Entries', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

fig.suptitle('CISA KEV + EPSS: What Exploitable Actually Looks Like', fontsize=13, fontweight='bold')
fig.text(0.5, 0.02, 'Source: CISA KEV catalog + api.first.org EPSS + NVD API',
         ha='center', fontsize=8, color='#666', style='italic', transform=fig.transFigure)
plt.savefig(OUTPUT / '02_kev_epss_analysis.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

## 4. Real Triage Query: Prioritise a CVE List

This is what you'd actually build for a security operations workflow. Given any list of CVE IDs, it queries KEV membership and EPSS scores, then returns a priority-ordered DataFrame.

The sample list below uses real recent CVEs — feel free to replace with CVEs relevant to your environment.

In [ ]:
# --- PARAMETER: replace with CVEs from your own environment ---
MY_CVE_LIST = [
    'CVE-2021-44228',  # Log4Shell
    'CVE-2022-26134',  # Confluence RCE
    'CVE-2023-44487',  # HTTP/2 Rapid Reset
    'CVE-2023-20198',  # Cisco IOS XE
    'CVE-2024-3400',   # PAN-OS command injection
    'CVE-2024-21762',  # Fortinet
    'CVE-2023-23397',  # Outlook NTLM
    'CVE-2020-1472',   # Zerologon
    'CVE-2019-19781',  # Citrix ADC
    'CVE-2024-27198',  # JetBrains TeamCity
    'CVE-2023-4966',   # Citrix Bleed
    'CVE-2022-3236',   # Sophos Firewall
    'CVE-2024-6387',   # regreSSHion
    'CVE-2023-35078',  # Ivanti MobileIron
    'CVE-2017-0144',   # EternalBlue
]


def triage_cves(cve_ids: list, kev_df: pd.DataFrame) -> pd.DataFrame:
    """Priority-order a CVE list using KEV membership and EPSS scores."""
    kev_set = set(kev_df['cveID'].values)
    
    # Fetch EPSS for the input list
    epss_scores = fetch_epss_batch(cve_ids)
    
    rows = []
    for cve_id in cve_ids:
        in_kev   = cve_id in kev_set
        epss_val = epss_scores.get(cve_id, None)
        if in_kev:
            kev_row  = kev_df[kev_df['cveID'] == cve_id].iloc[0]
            due_date = kev_row.get('dueDate', 'N/A')
            product  = f"{kev_row['vendorProject']} — {kev_row['product']}"
        else:
            due_date = 'N/A'
            product  = 'N/A'
        rows.append({'cve_id': cve_id, 'in_kev': in_kev,
                     'epss_pct': round(epss_val * 100, 2) if epss_val else None,
                     'kev_due_date': due_date, 'product': product})
    
    result = pd.DataFrame(rows)
    # Sort: KEV first, then by EPSS descending
    result = result.sort_values(['in_kev', 'epss_pct'], ascending=[False, False])
    result['tier'] = result.apply(
        lambda r: 'P1 — KEV' if r['in_kev'] else
                  ('P2 — EPSS>10%' if (r['epss_pct'] or 0) > 10 else
                   ('P3 — EPSS>1%' if (r['epss_pct'] or 0) > 1 else 'P4 — Monitor')),
        axis=1)
    return result.reset_index(drop=True)


print('Running triage query...')
triage_result = triage_cves(MY_CVE_LIST, kev)

pd.set_option('display.max_colwidth', 45)
pd.set_option('display.width', 120)
print('\nPriority-ordered triage results:')
print(triage_result[['cve_id', 'tier', 'epss_pct', 'in_kev', 'kev_due_date', 'product']].to_string(index=False))

## 5. The Noise-to-Signal Ratio by Tier

Given the 2024 cohort (~40,000 new CVEs), how many fall into each tier? This is the core of the "rain vs. floods" argument.

In [ ]:
# Tier counts derived from real KEV data + EPSS distribution analysis
ANNUAL_CVE_2024 = 40_000  # NVD/MITRE published total for 2024

# These are empirical estimates derived from:
# - KEV additions per year: ~150-300 (using our live data)
# - EPSS > 10% historically captures ~10-15% of CVEs (live EPSS data shows distribution)
# - The EPSS % calculation below is approximate since we're using a sample

kev_new_2024 = len(kev[kev['year_added'] == 2024]) if 2024 in kev['year_added'].values else 196

# Approximate annual counts per tier based on EPSS distribution research
# Source: FIRST.org EPSS analysis papers
tier_data = {
    'P1 — KEV only':           kev_new_2024,
    'P2 — KEV + EPSS > 10%':  int(ANNUAL_CVE_2024 * 0.10),   # ~10% of CVEs exceed 10% EPSS
    'P3 — EPSS > 1%':         int(ANNUAL_CVE_2024 * 0.35),   # ~35% exceed 1%
    'P4 — All CVEs':           ANNUAL_CVE_2024,
}

print(f'Noise-to-signal analysis for 2024 annual CVE cohort ({ANNUAL_CVE_2024:,} CVEs):\n')
print(f'{"Tier":<30} {"CVE count":>10} {"% of total":>12} {"Noise ratio":>14}')
print('-' * 70)
for tier, count in tier_data.items():
    pct   = count / ANNUAL_CVE_2024 * 100
    ratio = ANNUAL_CVE_2024 / count
    print(f'{tier:<30} {count:>10,} {pct:>11.1f}% {ratio:>12.0f}:1')

print(f'\nKEV entries added in 2024 (live from catalog): {kev_new_2024}')
print(f'Noise-to-signal at KEV tier: {ANNUAL_CVE_2024 / kev_new_2024:.0f}:1')
print('\n→ Patching KEV entries only requires acting on {:.1f}% of the annual CVE feed.'.format(
    kev_new_2024 / ANNUAL_CVE_2024 * 100))